In [15]:
%pip install javalang neo4j


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: /Users/sriram-14910/code/code-rag/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
#Python 3 kernel required

from neo4j import GraphDatabase
URI = "neo4j://127.0.0.1:7687"  
AUTH = ("neo4j", "12345678")

# --- Connect to the database ---
driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connection to Neo4j successful!")
SOURCE_FOLDER = "/Users/sriram-14910/code/cache_framework/source" #needs to be changed
EXPORT_FOLDER = "/Users/sriram-14910/code/code-rag/code_representation_graph" 

Connection to Neo4j successful!


In [18]:
import os
import javalang
from neo4j import GraphDatabase
import datetime

#Here, the code model is also perissted as a file, I initially did this for static analyis and it can be skipped.
# ❗ CRITICAL: Update this path to your Neo4j database's 'import' directory
NEO4J_IMPORT_DIR = "/Users/sriram-14910/Library/Application Support/neo4j-desktop/Application/Data/dbmss/dbms-22f22018-8060-4834-a95a-5c69172a0293" 

# This is the sub-folder we will create inside the import directory
EXPORT_FOLDER = "codebase_exports" 

# --- Database Connection ---
driver = GraphDatabase.driver(URI, auth=AUTH)
try:
    driver.verify_connectivity()
    print("✅ Connection to Neo4j successful!")
except Exception as e:
    print(f"❌ Could not connect to Neo4j: {e}")
    exit()

# --- 📄 2. PARSING LOGIC ---
def analyze_java_files(source_folder):
    """
    Parses all Java files and returns a structured list of dictionaries.
    """
    codebase_data = []
    if not os.path.exists(source_folder):
        print(f"❌ Source folder not found: {source_folder}")
        return None

    for root, _, files in os.walk(source_folder):
        for file in files:
            if file.endswith(".java"):
                file_path = os.path.join(root, file)
                print(f"Parsing: {file_path}")
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        code = f.read()
                    
                    tree = javalang.parse.parse(code)
                    package_name = tree.package.name if tree.package else "default"
                    imports = [imp.path for imp in tree.imports]

                    for type_node in tree.types:
                        type_details = {
                            'file_path': file_path, 'package': package_name, 'imports': imports,
                            'name': type_node.name, 'type': 'Class', 'fields': [], 'methods': []
                        }

                        if isinstance(type_node, javalang.tree.InterfaceDeclaration):
                            type_details['type'] = 'Interface'
                        elif isinstance(type_node, javalang.tree.EnumDeclaration):
                            type_details['type'] = 'Enum'
                        
                        if hasattr(type_node, 'fields'):
                            type_details['fields'] = [{'name': f.declarators[0].name, 'type': f.type.name} for f in type_node.fields]
                        
                        if hasattr(type_node, 'methods'):
                            for method_node in type_node.methods:
                                signature = f"{type_node.name}.{method_node.name}({', '.join(p.type.name for p in method_node.parameters)})"
                                method_details = {
                                    'name': method_node.name, 'signature': signature,
                                    'parameters': [{'name': p.name, 'type': p.type.name} for p in method_node.parameters],
                                    'calls': []
                                }
                                if method_node.body:
                                    for statement in method_node.body:
                                        for _, call_node in statement.filter(javalang.tree.MethodInvocation):
                                            method_details['calls'].append(call_node.member)
                                type_details['methods'].append(method_details)
                        
                        codebase_data.append(type_details)
                except Exception as e:
                    print(f"⚠️ Failed to parse {file_path}: {e}")
    return codebase_data

# --- 💾 3. NEO4J LOADING & EXPORT LOGIC ---
def load_single_item_to_neo4j(tx, item_data, timestamp):
    """
    Processes a SINGLE parsed item. The item is wrapped in a list
    so the UNWIND-based queries can be reused.
    """
    # The queries are the same, but they will now operate on a list containing only one item.
    data = [item_data]
    node_creation_query = """
    UNWIND $data as row
    MERGE (p:Package {name: row.package})
    MERGE (f:File {path: row.file_path}) ON CREATE SET f.createdAt = $timestamp ON MATCH SET f.lastSeen = $timestamp
    MERGE (t {name: row.name}) ON CREATE SET t.createdAt = $timestamp ON MATCH SET t.lastSeen = $timestamp
    SET t.file_path = row.file_path, t:`{type}`
    MERGE (p)-[:CONTAINS]->(t)
    MERGE (f)-[:CONTAINS_TYPE]->(t)
    FOREACH (method IN row.methods |
        MERGE (m:Method {signature: method.signature}) ON CREATE SET m.createdAt = $timestamp ON MATCH SET m.lastSeen = $timestamp
        SET m.name = method.name
        MERGE (t)-[:HAS_METHOD]->(m)
    )
    """
    relationship_creation_query = """
    UNWIND $data as row
    UNWIND row.methods as method
    UNWIND method.calls as call_name
    MATCH (caller:Method {signature: method.signature})
    MATCH (callee:Method {name: call_name})
    MERGE (caller)-[:CALLS]->(callee)
    """
    # The item_data dictionary already contains the 'type'
    label = item_data['type']
    specific_query = node_creation_query.replace('`{type}`', label)
    tx.run(specific_query, data=data, timestamp=timestamp)
    tx.run(relationship_creation_query, data=data)

def cleanup_old_nodes(tx, timestamp):
    """Finds any nodes not seen in the latest run and deletes them."""
    tx.run("MATCH (n) WHERE n.lastSeen < $timestamp DETACH DELETE n", timestamp=timestamp)

def export_graph_to_graphml(tx, file_name):
    """Exports the entire database to a GraphML file using APOC."""
    tx.run("CALL apoc.export.graphml.all($file, {useTypes: true})", file=file_name)

# --- 🚀 4. MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    run_timestamp = datetime.datetime.now()
    start_time_iso = run_timestamp.isoformat()
    
    parsed_data = analyze_java_files(SOURCE_FOLDER)
    
    if parsed_data:
        # --- ✨ FIX: No batching. Loop through each item individually. ---
        print(f"\nLoading {len(parsed_data)} parsed types to Neo4j one by one...")
        total_items = len(parsed_data)
        for i, item in enumerate(parsed_data):
            with driver.session(database="neo4j") as session:
                session.execute_write(load_single_item_to_neo4j, item, start_time_iso)
                print(f"  ✅ Loaded item {i + 1}/{total_items} in its own transaction.")
        print("✅ Load complete.")

        print("\nCleaning up old nodes...")
        with driver.session(database="neo4j") as session:
            session.execute_write(cleanup_old_nodes, start_time_iso)
        print("✅ Cleanup complete.")

        try:
            export_dir_path = os.path.join(NEO4J_IMPORT_DIR, EXPORT_FOLDER)
            os.makedirs(export_dir_path, exist_ok=True)
            print(f"\n✅ Ensured export directory exists: {export_dir_path}")

            timestamp_str = run_timestamp.strftime("%Y%m%d_%H%M%S")
            export_filename_for_apoc = f"{EXPORT_FOLDER}/code_graph_{timestamp_str}.graphml"
            
            print(f"Exporting graph to {export_filename_for_apoc}...")
            with driver.session(database="neo4j") as session:
                session.execute_write(export_graph_to_graphml, export_filename_for_apoc)
            print("✅ Export complete.") #this can be skipped.

        except Exception as e:
            print(f"❌ An error occurred during export: {e}")
            print("Please ensure the NEO4J_IMPORT_DIR path is correct and you have write permissions.")

    driver.close()
    print("\nScript finished successfully.")

✅ Connection to Neo4j successful!
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICServer.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICStatusCode.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICProcessUtil.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICServerException.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICService.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICConstants.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICProviderImpl.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICProvider.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICDeploymentUtil.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/ZICNetworkUtil.java
Par

Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/api/impl/ZICServerAPIImpl.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/api/impl/ZICAppAPICacheImpl.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/data/LongSequenceGenerator.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/data/ResourceAuditTransferInterfaceImpl.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/data/ZICRESTHandlerImpl.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/data/handler/CreatedByHandler.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/data/handler/ClusterSyncHandler.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/data/handler/TimeStampHandler.java
Parsing: /Users/sriram-14910/code/cache_framework/source/server/com/zoho/zic/data/handler/AppHandler.java
P

Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/connection/JedisConnectionImpl.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/APIException.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/DefaultArgumentConverter.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/DefaultMethodHasher.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/ZICAPIException.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/APIProxy.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/APIConstants.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/ArgumentConverter.java
Parsing: /Users/sriram-14910/code/cache_framework/source/agent/com/zoho/zic/oncloud/api/MethodHasher.java
Parsing: 

  ✅ Loaded item 150/211 in its own transaction.
  ✅ Loaded item 151/211 in its own transaction.
  ✅ Loaded item 152/211 in its own transaction.
  ✅ Loaded item 153/211 in its own transaction.
  ✅ Loaded item 154/211 in its own transaction.
  ✅ Loaded item 155/211 in its own transaction.
  ✅ Loaded item 156/211 in its own transaction.
  ✅ Loaded item 157/211 in its own transaction.
  ✅ Loaded item 158/211 in its own transaction.
  ✅ Loaded item 159/211 in its own transaction.
  ✅ Loaded item 160/211 in its own transaction.
  ✅ Loaded item 161/211 in its own transaction.
  ✅ Loaded item 162/211 in its own transaction.
  ✅ Loaded item 163/211 in its own transaction.
  ✅ Loaded item 164/211 in its own transaction.
  ✅ Loaded item 165/211 in its own transaction.
  ✅ Loaded item 166/211 in its own transaction.
  ✅ Loaded item 167/211 in its own transaction.
  ✅ Loaded item 168/211 in its own transaction.
  ✅ Loaded item 169/211 in its own transaction.
  ✅ Loaded item 170/211 in its own trans

In [11]:
this is a python3 virtual kernel that i have created in the project directory and have used as the kernel here.


the notebook will load the AST onto the graph database and now the data is setup.

now you need to run run.sh

initial run will have the NL to DSL model training run, so it will be pretty slow(1 hour on M1 macs)

once it is run, the startup will take ~5 seconds.

SyntaxError: invalid syntax (3797507595.py, line 1)

In [10]:
you just need to run all of the cells. after replacing directories.

SyntaxError: invalid syntax (1516087794.py, line 1)

In [ ]:
the final step exports data as a graphML file for static analysis - other purposes. you can skip this step if you want